In [117]:
pip install ucimlrepo

In [118]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix


In [119]:
# fetch dataset
student_performance = fetch_ucirepo(id=320)

# data (as pandas dataframes)
X = student_performance.data.features
y = student_performance.data.targets

# metadata
print(student_performance.metadata)

# variable information
print(student_performance.variables)

{'uci_id': 320, 'name': 'Student Performance', 'repository_url': 'https://archive.ics.uci.edu/dataset/320/student+performance', 'data_url': 'https://archive.ics.uci.edu/static/public/320/data.csv', 'abstract': 'Predict student performance in secondary education (high school). ', 'area': 'Social Science', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Multivariate'], 'num_instances': 649, 'num_features': 30, 'feature_types': ['Integer'], 'demographics': ['Sex', 'Age', 'Other', 'Education Level', 'Occupation'], 'target_col': ['G1', 'G2', 'G3'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2008, 'last_updated': 'Fri Jan 05 2024', 'dataset_doi': '10.24432/C5TG7T', 'creators': ['Paulo Cortez'], 'intro_paper': {'ID': 360, 'type': 'NATIVE', 'title': 'Using data mining to predict secondary school student performance', 'authors': 'P. Cortez, A. M. G. Silva', 'venue': 'Proceedings of 5th Annual Future Business Technolo

# 1. ESTATÍSTICAS DESCRITIVAS

In [120]:
df = pd.concat([X, y], axis=1)
print("=== Estatísticas Descritivas ===")
display(df.describe(include='all'))

=== Estatísticas Descritivas ===


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
count,649,649,649.000000,649,649,649,649.000000,649.000000,649,649,...,649.000000,649.000000,649.000000,649.000000,649.000000,649.000000,649.000000,649.000000,649.000000,649.000000
unique,2,2,NaN,2,2,2,NaN,NaN,5,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,GP,F,NaN,U,GT3,T,NaN,NaN,other,other,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,423,383,NaN,452,457,569,NaN,NaN,258,367,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,16.744222,NaN,NaN,NaN,2.514638,2.306626,NaN,NaN,...,3.930663,3.180277,3.184900,1.502311,2.280431,3.536210,3.659476,11.399076,11.570108,11.906009
std,NaN,NaN,1.218138,NaN,NaN,NaN,1.134552,1.099931,NaN,NaN,...,0.955717,1.051093,1.175766,0.924834,1.284380,1.446259,4.640759,2.745265,2.913639,3.230656
min,NaN,NaN,15.000000,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,NaN,NaN,16.000000,NaN,NaN,NaN,2.000000,1.000000,NaN,NaN,...,4.000000,3.000000,2.000000,1.000000,1.000000,2.000000,0.000000,10.000000,10.000000,10.000000
50%,NaN,NaN,17.000000,NaN,NaN,NaN,2.000000,2.000000,NaN,NaN,...,4.000000,3.000000,3.000000,1.000000,2.000000,4.000000,2.000000,11.000000,11.000000,12.000000
75%,NaN,NaN,18.000000,NaN,NaN,NaN,4.000000,3.000000,NaN,NaN,...,5.000000,4.000000,4.000000,2.000000,3.000000,5.000000,6.000000,13.000000,13.000000,14.000000


# 2. TRANSFORMAÇÕES DE LINHA E COLUNA

In [121]:
df = df[df['absences'] <= 30]
df['Target'] = (df['G3'] >= 6).astype(int)
df = df.drop(columns=['G1', 'G2', 'G3'])
df = pd.get_dummies(df, drop_first=True)

# 3. DIVISÃO EM TRÊS SUBCONJUNTOS

In [116]:
X = df.drop('Target', axis=1)
y = df['Target']
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)


# 4. TREINAMENTO E AVALIAÇÃO DO MODELO

In [122]:
modelo = RandomForestClassifier(random_state=42)
modelo.fit(X_train, y_train)

pred_val = modelo.predict(X_val)
print(f"\nAcurácia na Validação: {accuracy_score(y_val, pred_val):.2f}")

pred_test = modelo.predict(X_test)
print(f"Acurácia no Teste: {accuracy_score(y_test, pred_test):.2f}")

print("\n=== Matriz de Confusão ===")
print(confusion_matrix(y_test, pred_test))


Acurácia na Validação: 0.97
Acurácia no Teste: 0.99

=== Matriz de Confusão ===
[[ 0  1]
 [ 0 97]]


# 5. PREDIÇÃO DO MODELO IMPLANTADO

In [123]:
amostra = X_test.iloc[[0]]
resultado = "Aprovado" if modelo.predict(amostra)[0] == 1 else "Reprovado"
print(f"\n=== Resultado de Predição ===")
print(f"O aluno de teste foi classificado como: {resultado}")


=== Resultado de Predição ===
O aluno de teste foi classificado como: Aprovado
